# 06 — Mechanistic Analysis

**Notebook 6 of 7.** Turns the report from "we measured accuracy" into "we measured accuracy AND characterized what changed inside the model."

Three probes on the trained LoRA adapters (uses **seed 1** of each condition — the mechanistic story shouldn't be seed-sensitive):

- **Probe A — KL divergence per token position.** Where in a GSM8K solution did each condition change the model's next-token distribution most? Teacher-forces base-generated sequences through each adapter and computes KL(base || adapter) at each position, averaged over 50 held-out problems.
- **Probe B — LoRA weight-norm audit.** For each LoRA-targeted module (7 modules × 28 layers = 196 per condition), how much did the adapter's effective weight delta (‖B·A‖_F) move that projection? Grouped by layer and module type.
- **Probe C — Discordant-problem structural coding.** For problems where any DPO condition disagreed with baseline, code the flip pattern and structural features (response length, completion status, arithmetic-operation count).

- **Runtime:** ~15 minutes on Colab free-tier T4.
- **Prerequisites:** Notebooks 03 and 04 with `SEED=1` have run (they save adapters to `adapters/dpo_{A,B}_seed1/` on Drive).
- **Outputs on Drive:**
  - `results/figures/kl_by_position.png`
  - `results/figures/weight_norm_deltas.png`
  - `results/figures/discordant_analysis.png`
  - `results/mechanistic_stats.json`

## 1. Setup

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
from pathlib import Path

PROJECT_ROOT = '/content/drive/MyDrive/dpo-reasoning-transfer'
os.makedirs(f'{PROJECT_ROOT}/results/figures', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/results/mechanistic', exist_ok=True)

CODE_DIR = PROJECT_ROOT   # or f'{PROJECT_ROOT}/repo' if cloned from GitHub
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.stats import load_all_results, per_problem_correctness
from src.eval import GSM8K_PROMPT
print('src modules loaded')

## 2. Prerequisites

Verifies that both seed-1 adapters exist. If not, prints exact commands to fix it and halts. Notebook 06 uses seed 1 by convention — the mechanistic story shouldn't depend on training seed, and running the analysis for all 3 seeds would triple the compute time for marginal added value.

In [ ]:
ADAPTER_A_PATH = f'{PROJECT_ROOT}/adapters/dpo_A_seed1'
ADAPTER_B_PATH = f'{PROJECT_ROOT}/adapters/dpo_B_seed1'

for name, path in [('Condition A adapter', ADAPTER_A_PATH), ('Condition B adapter', ADAPTER_B_PATH)]:
    manifest = Path(path) / 'adapter_model.safetensors'
    manifest_alt = Path(path) / 'adapter_model.bin'
    if not (manifest.exists() or manifest_alt.exists()):
        print(f'MISSING: {name} at {path}')
        print(f'  Fix: re-run Notebook {"03" if "A" in name else "04"} with SEED=1 '
              f'(the updated version saves adapters to this location).')
        raise SystemExit(f'Missing prerequisite: {path}')

print(f'Found: {ADAPTER_A_PATH}')
print(f'Found: {ADAPTER_B_PATH}')

# Sanity-check the provenance files
for path in (ADAPTER_A_PATH, ADAPTER_B_PATH):
    prov_path = Path(path) / 'provenance.json'
    if prov_path.exists():
        with open(prov_path) as f:
            prov = json.load(f)
        print(f"  {path}: {prov.get('condition')} seed={prov.get('seed')} "
              f"(pref set: {prov.get('preference_dataset', 'unknown')})")

## 3. Load base model with both adapters attached

PEFT supports multiple named adapters on one base model. We attach both, then switch between them (and use `disable_adapter()` for the base) to get three "modes" from a single 4-bit model in ~4GB of VRAM.

In [ ]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

# Attach adapter A first (creates the PeftModel wrapper), then load B onto it
model = PeftModel.from_pretrained(base_model, ADAPTER_A_PATH, adapter_name='A')
model.load_adapter(ADAPTER_B_PATH, adapter_name='B')

print(f'Loaded adapters: {list(model.peft_config.keys())}')
print(f'Active adapter (default): {model.active_adapters}')
print(f'Device: {next(model.parameters()).device}')

## 4. Probe B — LoRA weight-norm audit

Cheap and informative: for each LoRA-adapted module, compute ‖B·A‖_F where A and B are the low-rank factors. This is the Frobenius norm of the effective weight delta the adapter applies. Report broken down by (layer_index, module_type). A striking pattern would be Condition B concentrating mass in later MLP layers (which do computation) while Condition A spreads mass thinner across attention layers (which do routing).

Doing this first because it needs no forward passes — pure weight inspection.

In [ ]:
import re
import numpy as np

def layer_index_from_name(module_name: str) -> int | None:
    m = re.search(r'layers\.(\d+)\.', module_name)
    return int(m.group(1)) if m else None

def module_type_from_name(module_name: str) -> str:
    for t in ['q_proj', 'k_proj', 'v_proj', 'o_proj',
              'gate_proj', 'up_proj', 'down_proj']:
        if module_name.endswith(t) or f'.{t}.' in module_name:
            return t
    return 'other'

def collect_lora_norms(peft_model, adapter_name: str) -> list[dict]:
    """For each LoRA-adapted submodule, compute ‖B·A‖_F for the named adapter."""
    records = []
    for name, mod in peft_model.named_modules():
        # PEFT LoRA layers have `lora_A` and `lora_B` as ModuleDicts keyed by adapter name
        if not (hasattr(mod, 'lora_A') and hasattr(mod, 'lora_B')):
            continue
        if adapter_name not in mod.lora_A:
            continue
        A = mod.lora_A[adapter_name].weight.detach().float()   # (r, in_dim)
        B = mod.lora_B[adapter_name].weight.detach().float()   # (out_dim, r)
        delta = B @ A                                          # (out_dim, in_dim)
        records.append({
            'module_name': name,
            'layer': layer_index_from_name(name),
            'kind': module_type_from_name(name),
            'r': A.shape[0],
            'in_dim': A.shape[1],
            'out_dim': B.shape[0],
            'fro_norm': float(delta.norm(p='fro').item()),
            'fro_norm_per_element': float(delta.norm(p='fro').item()
                                          / (A.shape[1] * B.shape[0]) ** 0.5),
        })
    return records

norms_A = collect_lora_norms(model, 'A')
norms_B = collect_lora_norms(model, 'B')
print(f'Condition A: {len(norms_A)} LoRA modules audited')
print(f'Condition B: {len(norms_B)} LoRA modules audited')
print()
print(f"Total ‖B·A‖_F summed across modules:")
print(f"  Condition A: {sum(r['fro_norm'] for r in norms_A):.2f}")
print(f"  Condition B: {sum(r['fro_norm'] for r in norms_B):.2f}")

### 4.1 Weight-norm figure — grouped by module type and layer

Two views: (a) per-module-type totals summed across layers, and (b) per-layer trajectories for the most-moved module type. The per-element normalization removes the size-of-projection confound (some modules are bigger than others).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MODULE_TYPES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']

def per_type_totals(records):
    totals = {t: [] for t in MODULE_TYPES}
    for r in records:
        if r['kind'] in totals:
            totals[r['kind']].append(r['fro_norm_per_element'])
    return {t: np.mean(v) if v else 0.0 for t, v in totals.items()}

per_type_A = per_type_totals(norms_A)
per_type_B = per_type_totals(norms_B)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) per-module-type mean of per-element Fro norm
x = np.arange(len(MODULE_TYPES))
w = 0.35
axes[0].bar(x - w/2, [per_type_A[t] for t in MODULE_TYPES], w,
            label='Condition A (UltraFeedback)', color='#4C72B0', alpha=0.85)
axes[0].bar(x + w/2, [per_type_B[t] for t in MODULE_TYPES], w,
            label='Condition B (self-gen reasoning)', color='#DD8452', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(MODULE_TYPES, rotation=25, ha='right')
axes[0].set_ylabel('mean ‖B·A‖_F per element')
axes[0].set_title('LoRA weight change by module type\n(averaged across all layers)')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_axisbelow(True)

# (b) per-layer trajectory for the most-moved module type (union of both conditions)
top_type = max(MODULE_TYPES, key=lambda t: max(per_type_A[t], per_type_B[t]))

def per_layer_for_type(records, module_type):
    by_layer = {}
    for r in records:
        if r['kind'] == module_type and r['layer'] is not None:
            by_layer[r['layer']] = r['fro_norm_per_element']
    return by_layer

by_layer_A = per_layer_for_type(norms_A, top_type)
by_layer_B = per_layer_for_type(norms_B, top_type)
layers = sorted(set(by_layer_A) | set(by_layer_B))
axes[1].plot(layers, [by_layer_A.get(l, 0) for l in layers],
             marker='o', color='#4C72B0', label='Condition A', linewidth=1.5, markersize=5)
axes[1].plot(layers, [by_layer_B.get(l, 0) for l in layers],
             marker='s', color='#DD8452', label='Condition B', linewidth=1.5, markersize=5)
axes[1].set_xlabel('Transformer layer index')
axes[1].set_ylabel('‖B·A‖_F per element')
axes[1].set_title(f'{top_type} — weight change by layer\n'
                  f'(module type with the largest condition difference)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig_path = f'{PROJECT_ROOT}/results/figures/weight_norm_deltas.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 5. Probe A — KL divergence per token position

For each of 50 held-out GSM8K problems (test[500:550] — untouched by both training and eval):
1. Generate a response with the **base** model.
2. Teacher-force that response through the base and each adapter.
3. At each position, compute KL(base || adapter).
4. Aggregate mean KL by *relative* position (fractional depth within the response), so responses of different lengths are commensurable.

KL(base || A) tells us how much Condition A "pulled" the model at each token; comparing that curve to KL(base || B) shows whether one condition changes early tokens (planning) or late tokens (arithmetic) more.

In [ ]:
from datasets import load_dataset

# Held-out set: test[500:550] — the 500 problems in [0:500] are the eval set,
# these 50 have never been trained OR evaluated on.
heldout = load_dataset('openai/gsm8k', 'main', split='test[500:550]')
print(f'Held-out set: {len(heldout)} problems (GSM8K test[500:550])')

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def generate_with_base(prompt_text, max_new_tokens=150):
    """Greedy-generate with the base model (adapters disabled)."""
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with model.disable_adapter():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Returns: full token ids (prompt + generation), and the prompt length
    return out[0], inputs['input_ids'].shape[1]

@torch.no_grad()
def kl_per_position(logits_p, logits_q):
    """KL(P || Q) per position. Both shape (seq_len, vocab). Returns (seq_len,)."""
    log_p = F.log_softmax(logits_p, dim=-1)
    log_q = F.log_softmax(logits_q, dim=-1)
    p = log_p.exp()
    return (p * (log_p - log_q)).sum(dim=-1)

def prompt_for(question):
    messages = [{'role': 'user', 'content': GSM8K_PROMPT.format(question=question)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print('KL utilities defined.')

In [ ]:
from tqdm.auto import tqdm

# Per-problem KL results
per_problem_kl = []   # list of dicts

for i, ex in enumerate(tqdm(heldout, desc='KL analysis')):
    prompt_text = prompt_for(ex['question'])

    # Generate with base
    full_ids, prompt_len = generate_with_base(prompt_text, max_new_tokens=150)
    response_len = full_ids.shape[0] - prompt_len
    if response_len < 5:
        continue    # skip degenerate cases

    # Get logits from all three modes on the same reference sequence.
    # For a next-token prediction, position t's logits predict token t+1;
    # we care about positions [prompt_len-1 : -1] because those predict the response tokens.
    with model.disable_adapter():
        logits_base = model(input_ids=full_ids.unsqueeze(0)).logits[0]

    model.set_adapter('A')
    logits_A = model(input_ids=full_ids.unsqueeze(0)).logits[0]

    model.set_adapter('B')
    logits_B = model(input_ids=full_ids.unsqueeze(0)).logits[0]

    # Response-predicting positions
    resp_slice = slice(prompt_len - 1, full_ids.shape[0] - 1)

    kl_A = kl_per_position(logits_base[resp_slice], logits_A[resp_slice]).cpu().float().numpy()
    kl_B = kl_per_position(logits_base[resp_slice], logits_B[resp_slice]).cpu().float().numpy()

    per_problem_kl.append({
        'index': i,
        'response_len': response_len,
        'kl_A': kl_A.tolist(),
        'kl_B': kl_B.tolist(),
    })

    # Free memory each iteration
    del logits_base, logits_A, logits_B
    torch.cuda.empty_cache()

print(f'Computed KL for {len(per_problem_kl)}/{len(heldout)} held-out problems')

### 5.1 KL curves by relative position

Interpolate each problem's KL trace to a fixed 20-bin relative-position grid, then average. Lets us see WHERE in a solution each condition changes the model most.

In [ ]:
N_BINS = 20

def resample_to_bins(kl_values, n_bins=N_BINS):
    """Interpolate a variable-length KL trace onto n_bins equal-width relative positions."""
    kl = np.asarray(kl_values, dtype=float)
    if len(kl) < 2:
        return np.full(n_bins, np.nan)
    x_src = np.linspace(0, 1, len(kl))
    x_dst = np.linspace(0, 1, n_bins)
    return np.interp(x_dst, x_src, kl)

kl_A_resampled = np.stack([resample_to_bins(r['kl_A']) for r in per_problem_kl])
kl_B_resampled = np.stack([resample_to_bins(r['kl_B']) for r in per_problem_kl])

# Mean and SE across problems
def mean_and_sem(arr):
    m = np.nanmean(arr, axis=0)
    n = (~np.isnan(arr)).sum(axis=0)
    se = np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.maximum(n, 1))
    return m, se

mean_A, se_A = mean_and_sem(kl_A_resampled)
mean_B, se_B = mean_and_sem(kl_B_resampled)

# Figure
fig, ax = plt.subplots(figsize=(9, 4.5))
rel_pos = np.linspace(0, 1, N_BINS)

ax.plot(rel_pos, mean_A, color='#4C72B0', lw=2, marker='o', markersize=5,
        label='Condition A (UltraFeedback)')
ax.fill_between(rel_pos, mean_A - se_A, mean_A + se_A, color='#4C72B0', alpha=0.2)

ax.plot(rel_pos, mean_B, color='#DD8452', lw=2, marker='s', markersize=5,
        label='Condition B (self-gen reasoning)')
ax.fill_between(rel_pos, mean_B - se_B, mean_B + se_B, color='#DD8452', alpha=0.2)

ax.set_xlabel('Relative position within response (0 = start, 1 = end)')
ax.set_ylabel('KL(base || adapter) per token, nats')
ax.set_title(f'Token-distribution shift induced by each DPO condition\n'
             f'Mean ± 1 SE across {len(per_problem_kl)} held-out GSM8K problems')
ax.legend()
ax.grid(alpha=0.3)

# Annotate summary numbers
ax.text(0.02, 0.98,
        f'Total mean KL:\n  A: {mean_A.mean():.3f} nats\n  B: {mean_B.mean():.3f} nats',
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(facecolor='white', alpha=0.85, edgecolor='#DDDDDD'))

plt.tight_layout()
fig_path = f'{PROJECT_ROOT}/results/figures/kl_by_position.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# Free the model — remaining probes are CPU-only
del model, base_model
torch.cuda.empty_cache()
print('GPU memory freed for CPU-only Probe C.')

## 6. Probe C — Discordant-problem structural analysis

Reads the eval JSONs directly (no GPU needed) and categorizes flipped problems by:
- **Flip pattern:** which conditions moved on this problem?
- **Response-length change:** paired chars-delta vs baseline.
- **Completion:** did the response end naturally (final answer) or hit the token cap?

The stacked bars answer: *Of the problems where DPO caused a change, what kind of change was it?*

In [ ]:
all_results = load_all_results(PROJECT_ROOT)
b_correct = per_problem_correctness(all_results['baseline'])
A_correct_seeds = {s: per_problem_correctness(r) for s, r in all_results['A'].items()}
B_correct_seeds = {s: per_problem_correctness(r) for s, r in all_results['B'].items()}

n_problems = len(b_correct)
print(f'Baseline: {int(b_correct.sum())}/{n_problems}')
print(f'A seeds: ' + ', '.join(f'{s}={int(v.sum())}' for s, v in A_correct_seeds.items()))
print(f'B seeds: ' + ', '.join(f'{s}={int(v.sum())}' for s, v in B_correct_seeds.items()))

# Per-problem "robust wins/losses" — majority vote across seeds
def robust_bit(seeds_dict, threshold=2):
    stacked = np.stack(list(seeds_dict.values()))   # (n_seeds, n_problems)
    return (stacked.sum(axis=0) >= threshold).astype(int)

A_robust = robust_bit(A_correct_seeds)
B_robust = robust_bit(B_correct_seeds)

# Cross-tab: (baseline correctness, A robust, B robust) → count
from collections import Counter
categories = Counter()
for i in range(n_problems):
    key = (int(b_correct[i]), int(A_robust[i]), int(B_robust[i]))
    categories[key] += 1

print('\n(baseline, A_robust, B_robust) → count:')
for key in sorted(categories):
    b, a, x = key
    label = f'baseline={"R" if b else "W"}, A={"R" if a else "W"}, B={"R" if x else "W"}'
    print(f'  {label}:  {categories[key]}')

In [ ]:
# Focus on discordant problems: baseline disagrees with A_robust or B_robust
discordant_ix = [i for i in range(n_problems)
                 if (b_correct[i] != A_robust[i]) or (b_correct[i] != B_robust[i])]

print(f'Discordant problems (baseline vs robust A or robust B): {len(discordant_ix)}')

# For each discordant problem, categorize the flip pattern
def flip_label(b, a, x):
    if b == 0 and a == 1 and x == 1: return 'both_rescued'
    if b == 0 and a == 0 and x == 1: return 'B_only_rescued'
    if b == 0 and a == 1 and x == 0: return 'A_only_rescued'
    if b == 1 and a == 0 and x == 0: return 'both_broke'
    if b == 1 and a == 0 and x == 1: return 'A_only_broke'
    if b == 1 and a == 1 and x == 0: return 'B_only_broke'
    return 'other'

flip_categories = Counter()
for i in discordant_ix:
    flip_categories[flip_label(int(b_correct[i]), int(A_robust[i]), int(B_robust[i]))] += 1

print('\nFlip category counts:')
for cat in ['both_rescued', 'A_only_rescued', 'B_only_rescued',
            'both_broke', 'A_only_broke', 'B_only_broke', 'other']:
    print(f'  {cat}: {flip_categories.get(cat, 0)}')

# Response length shift for each category
baseline_by_ix = {r['index']: r for r in all_results['baseline']['results']}
A_seed1_by_ix  = {r['index']: r for r in all_results['A'][1]['results']}
B_seed1_by_ix  = {r['index']: r for r in all_results['B'][1]['results']}

def length_delta(baseline_r, treatment_r):
    return len(treatment_r['response']) - len(baseline_r['response'])

by_category_len_A = {cat: [] for cat in flip_categories}
by_category_len_B = {cat: [] for cat in flip_categories}
for i in discordant_ix:
    cat = flip_label(int(b_correct[i]), int(A_robust[i]), int(B_robust[i]))
    b_r = baseline_by_ix[i]
    by_category_len_A[cat].append(length_delta(b_r, A_seed1_by_ix[i]))
    by_category_len_B[cat].append(length_delta(b_r, B_seed1_by_ix[i]))

print('\nMedian response-length delta (chars, vs baseline; seed-1 responses):')
for cat in sorted(flip_categories, key=lambda c: -flip_categories[c]):
    a_med = np.median(by_category_len_A[cat]) if by_category_len_A[cat] else 0
    b_med = np.median(by_category_len_B[cat]) if by_category_len_B[cat] else 0
    print(f'  {cat}:  A: {a_med:+6.0f}   B: {b_med:+6.0f}   (n={flip_categories[cat]})')

In [ ]:
# Figure: stacked bar of flip categories
fig, ax = plt.subplots(figsize=(9, 4.5))

cat_order = ['B_only_rescued', 'both_rescued', 'A_only_rescued',
             'B_only_broke', 'both_broke', 'A_only_broke', 'other']
cat_display = {
    'B_only_rescued': 'B rescued\n(A did not)',
    'both_rescued':   'Both rescued',
    'A_only_rescued': 'A rescued\n(B did not)',
    'B_only_broke':   'B broke\n(A did not)',
    'both_broke':     'Both broke',
    'A_only_broke':   'A broke\n(B did not)',
    'other':          'Other',
}
colors = {
    'B_only_rescued': '#2E7D32',   # green — B wins
    'both_rescued':   '#66BB6A',
    'A_only_rescued': '#8BC34A',
    'B_only_broke':   '#C62828',   # red — B loses
    'both_broke':     '#EF5350',
    'A_only_broke':   '#FF9800',
    'other':          '#9E9E9E',
}

counts = [flip_categories.get(c, 0) for c in cat_order]
labels = [cat_display[c] for c in cat_order]
bar_colors = [colors[c] for c in cat_order]

bars = ax.bar(range(len(cat_order)), counts, color=bar_colors,
              edgecolor='black', linewidth=0.4, alpha=0.9)
for bar, n in zip(bars, counts):
    if n > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(n), ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_xticks(range(len(cat_order)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Number of problems')
ax.set_title('Where DPO changed the outcome vs. baseline\n'
             '(robust = ≥2 of 3 seeds agree)', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
fig_path = f'{PROJECT_ROOT}/results/figures/discordant_analysis.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 7. Save mechanistic summary

Machine-readable JSON that the report cites — same principle as `aggregated_stats.json` from Notebook 05.

In [ ]:
mechanistic_summary = {
    'weight_norms': {
        'A_total_fro': float(sum(r['fro_norm'] for r in norms_A)),
        'B_total_fro': float(sum(r['fro_norm'] for r in norms_B)),
        'A_per_type_mean_per_element': {t: float(per_type_A[t]) for t in MODULE_TYPES},
        'B_per_type_mean_per_element': {t: float(per_type_B[t]) for t in MODULE_TYPES},
        'top_type_by_condition_diff': top_type,
    },
    'kl_divergence': {
        'n_heldout_problems': len(per_problem_kl),
        'mean_kl_A_per_token': float(mean_A.mean()),
        'mean_kl_B_per_token': float(mean_B.mean()),
        'kl_A_by_bin': mean_A.tolist(),
        'kl_B_by_bin': mean_B.tolist(),
        'n_bins': N_BINS,
    },
    'discordant': {
        'n_discordant': len(discordant_ix),
        'flip_categories': dict(flip_categories),
        'median_length_delta_by_category_A': {
            c: float(np.median(v)) if v else 0.0
            for c, v in by_category_len_A.items()
        },
        'median_length_delta_by_category_B': {
            c: float(np.median(v)) if v else 0.0
            for c, v in by_category_len_B.items()
        },
    },
}

# Save also the raw per-module norms and per-problem KL traces for reproducibility
raw_dump = {
    'norms_A': norms_A,
    'norms_B': norms_B,
    'per_problem_kl': per_problem_kl,
}

summary_path = f'{PROJECT_ROOT}/results/mechanistic_stats.json'
with open(summary_path, 'w') as f:
    json.dump(mechanistic_summary, f, indent=2)
raw_path = f'{PROJECT_ROOT}/results/mechanistic/raw.json'
with open(raw_path, 'w') as f:
    json.dump(raw_dump, f, indent=2)

print(f'Saved: {summary_path}')
print(f'Saved: {raw_path}')
print()
print('=' * 70)
print('MECHANISTIC SUMMARY — paste into REPORT.md §6 Mechanistic analysis')
print('=' * 70)
print()
print(f'Weight norms (total ‖B·A‖_F summed across 196 modules):')
print(f"  Condition A: {mechanistic_summary['weight_norms']['A_total_fro']:.2f}")
print(f"  Condition B: {mechanistic_summary['weight_norms']['B_total_fro']:.2f}")
print(f"  Ratio B/A:   {mechanistic_summary['weight_norms']['B_total_fro'] / mechanistic_summary['weight_norms']['A_total_fro']:.2f}")
print()
print(f'KL divergence (mean per response token across {len(per_problem_kl)} held-out problems):')
print(f"  KL(base || A): {mean_A.mean():.4f} nats")
print(f"  KL(base || B): {mean_B.mean():.4f} nats")
print(f"  Ratio B/A:     {mean_B.mean() / max(mean_A.mean(), 1e-9):.2f}")
print()
print(f'Discordant-problem breakdown:')
for cat in cat_order:
    print(f"  {cat:20}: {flip_categories.get(cat, 0)}")

## Next

Everything the analytical side of the report needs is now on disk:
- Aggregated statistics: `results/aggregated_stats.json` (from Notebook 05)
- Mechanistic statistics: `results/mechanistic_stats.json` (from this notebook)
- Four figures under `results/figures/`

**Next up:** `README.md` and `REPORT.md` — the two write-ups that make the repo readable to a reviewer. Written last, when there are real numbers to describe.